# Random Forest (Pipeline B)

**Model**: RandomForest via `sklearn.ensemble.RandomForestRegressor`  
**Target**: `gdpc1`  
**Variables**: Cat3 (53 + COVID = 56)  
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025  
**Scaling**: NO (tree splits are scale-invariant)  
**HP tuning**: YES — GridSearchCV on {max_features, min_samples_leaf, n_estimators}  
**n_lags**: 4  
**10-model averaging**: Train 10 RFs with different seeds, average predictions. Reduces variance.  
**n_jobs**: 1 (Windows multiprocessing fix)

In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
                     get_features, split_for_scaler, load_data, PREDICTIONS_DIR, TARGET, COVID)

SEED = 42; np.random.seed(SEED)
TRAIN_START = "1959-01-01"; TRAIN_END = "2007-12-31"; VAL_END = "2016-12-31"
TEST_START = "2017-01-01"; TEST_END = "2025-12-31"
N_LAGS = 4; N_MODELS = 10
VINTAGES = {"m1": -2, "m2": -1, "m3": 0}
print("RF — Cat3, n_lags={}, scaling=NO, 10-model avg".format(N_LAGS))

RF — Cat3, n_lags=4, scaling=NO, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
print("Features: {}".format(len(features)))

# Phase A: HP tuning on validation window
tune_data = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how="any").reset_index(drop=True)
feature_cols = [c for c in tune_flat.columns if c != "date" and c != TARGET and any(c == f or c.startswith(f + "_") for f in features)]

tscv = TimeSeriesSplit(n_splits=5)
rf_base = RandomForestRegressor(random_state=SEED, n_jobs=1)
rf_grid = {"n_estimators": [200, 500], "max_depth": [None, 8, 16], "min_samples_leaf": [1, 3, 5]}
rf_search = GridSearchCV(rf_base, rf_grid, cv=tscv, scoring="neg_mean_squared_error", n_jobs=1)
rf_search.fit(tune_flat[feature_cols].values, tune_flat[TARGET].values)
best_params = rf_search.best_params_
print("Best RF params: {}".format(best_params))

Features: 56


Best RF params: {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 200}


In [3]:
test_quarters = monthly[(monthly["date"] >= TEST_START) & (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3,6,9,12])]["date"].tolist()
predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly["date"] == pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how="any").reset_index(drop=True)
        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue
        try:
            models = []
            for k in range(N_MODELS):
                m = RandomForestRegressor(
                    n_estimators=best_params["n_estimators"],
                    max_depth=best_params["max_depth"],
                    min_samples_leaf=best_params["min_samples_leaf"],
                    random_state=SEED+k, n_jobs=1)  # n_jobs=1: Windows multiprocessing fix
                m.fit(train_fl[feature_cols].values, train_fl[TARGET].values)
                models.append(m)
            tr = monthly[monthly["date"] == fc].reset_index(drop=True)
            tr_m = gen_lagged_data(metadata, tr.copy(), fc, lag=0)
            tr_f = mean_fill_dataset(train_m, tr_m)
            ctx = train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb = pd.concat([ctx, tr_f], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            tr_fl = cmb_fl.tail(1)  # last row = test row
            preds = [m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1) % 6 == 0:
        print("  {} / {}".format(i+1, len(test_quarters)))
print("Done. {} failures.".format(failed))

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

  36 / 36
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({"date": test_quarters, "actual": actuals_list, "prediction": predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, "rf_{}.csv".format(vn)), index=False)
    print("Saved rf_{}.csv".format(vn))

Saved rf_m1.csv


Saved rf_m2.csv
Saved rf_m3.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p)); return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum() > 0 else np.nan
def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p)); return np.mean(np.abs(a[m]-p[m])) if m.sum() > 0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print("RF best params: {}".format(best_params))
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}".format(vn, rmse(a[m], pp[m]), mae(a[m], pp[m])))

RF best params: {'max_depth': 8, 'min_samples_leaf': 3, 'n_estimators': 200}
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002731  MAE=0.002132
  m2  RMSFE=0.002768  MAE=0.002191
  m3  RMSFE=0.002739  MAE=0.002176
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042803  MAE=0.026710
  m2  RMSFE=0.042768  MAE=0.026678
  m3  RMSFE=0.042804  MAE=0.026714
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004505  MAE=0.003343
  m2  RMSFE=0.004513  MAE=0.003390
  m3  RMSFE=0.004523  MAE=0.003377
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019489  MAE=0.007969
  m2  RMSFE=0.019476  MAE=0.008003
  m3  RMSFE=0.019492  MAE=0.008000
